**Navigation** : [jumeau C#](ICT-49-JLensFideliteCrossTaille-CSharp.ipynb)

# ICT-49 — Stabilité cross-taille de la fidélité J-Lens

**Question.** Quand le modèle grossit de 1.7B à 27B, la lecture J-Lens reste-t-elle *stable* ? Concrètement : (1) les mêmes concepts sont-ils lus avec la même fidélité, et (2) la même structure de profondeur, à mesure que la taille augmente ?

Cet instrument répond en **CPU/numpy** sur les cinq échelles déjà commitées. Il prolonge le tête-à-tête SAE↔J-Lens (ICT-24) et la géométrie factored (ICT-36) en ajoutant l'axe de la **taille du modèle**.

## Garde-fous d'honnêteté

- **Substrat commité** : les fichiers `traces/calib_jlens_*.npz` (1.7B, 2B, 8B, 9B, 27B), produits par `scripts/extract_jlens_fidelity.py` (#8236(c)). Aucune exécution GPU ici — on lit des résumés de calibration déjà calculés.
- **Provenance des lens** : 1.7B / 2B = lens **local-fit** ; 8B / 9B / 27B = lens **publiés** (neuronpedia). La comparaison cross-taille mélange donc deux provenances — à garder en tête.
- **Petit échantillon** : `n_eval` = 4 prompts/concept. Les chiffres sont une **lecture indicative**, pas une mesure statistique.
- **Profondeurs alignées** : chaque échelle calibre ses couches en profondeur **fractionnelle** 0.25 / 0.50 (ex. 1.7B → 7/14 sur 28 ; 27B → 16/32 sur 64). On compare donc à profondeur fractionnelle égale, pas à couche absolue.

In [1]:
import os
import numpy as np
import pandas as pd

# Chemins relatifs au carnet : --cwd = repertoire de la serie ICT-Series
TRACES = "traces"
# 5 echelles, servables en CPU
SCALES = {
    "1.7B": "calib_jlens_qwen3-1-7b.npz",
    "2B":   "calib_jlens_qwen3-5-2b.npz",
    "8B":   "calib_jlens_qwen3-8b.npz",
    "9B":   "calib_jlens_qwen3-5-9b.npz",
    "27B":  "calib_jlens_qwen3-5-27b.npz",
}
ORDER = ["1.7B", "2B", "8B", "9B", "27B"]
METRICS = ["overlap10", "rel_l2", "kl"]

rows = []
for scale, fn in SCALES.items():
    d = np.load(os.path.join(TRACES, fn), allow_pickle=True)
    layers = d["meta_layers"].tolist()
    total = 2 * layers[-1]          # 2e couche en profondeur 0.50 -> total de couches
    for k in d.keys():
        if k.startswith("meta"):
            continue
        concept, _, layer = k.partition("__")
        ly = int(layer)
        v = d[k]                     # [overlap10, rel_l2, kl]
        rows.append({
            "scale": scale, "concept": concept, "layer": ly,
            "frac_depth": round(ly / total, 4),
            "overlap10": v[0], "rel_l2": v[1], "kl": v[2],
        })

df = pd.DataFrame(rows)
print(f"Table longue : {df.shape[0]} lignes, {df['scale'].nunique()} echelles, "
      f"{df['concept'].nunique()} concepts")
print("Profondeurs fractionnelles presentes :", sorted(df['frac_depth'].unique().tolist()))

Table longue : 50 lignes, 5 echelles, 5 concepts
Profondeurs fractionnelles presentes : [0.25, 0.5]


## 1. Tendance de fidélité par échelle

Trois métriques par (concept, couche) : `overlap@10` (plus **haut** = plus fidèle), `rel_l2` et `kl` (plus **bas** = plus fidèle). On les moyenne d'abord sur les 5 concepts × 2 profondeurs, par échelle.

In [2]:
by_scale = (df.groupby("scale")[METRICS].mean()
               .reindex(ORDER))
print("Fidelite moyenne par echelle (overlap10 : plus haut = plus fidele ; rel_l2/kl : plus bas = plus fidele)")
by_scale

Fidelite moyenne par echelle (overlap10 : plus haut = plus fidele ; rel_l2/kl : plus bas = plus fidele)


,overlap10,rel_l2,kl
scale,,,
1.7B,0.013776,0.767758,17.496718
2B,0.028427,1.230310,9.628378
8B,0.008420,1.590495,18.909991
9B,0.026780,0.964609,11.900528
27B,0.018645,1.114448,11.175254


In [3]:
pv = (df.pivot_table(index="concept", columns="scale",
                     values="rel_l2", aggfunc="mean")
          .reindex(columns=ORDER))
print("rel_l2 moyen par concept et echelle (plus bas = plus fidele)")
pv

rel_l2 moyen par concept et echelle (plus bas = plus fidele)


scale,1.7B,2B,8B,9B,27B
concept,,,,,
code_python,0.920388,1.299443,1.620747,1.048033,1.287340
dialogue,0.760831,1.269871,1.620004,0.932116,1.104762
math,0.766815,1.276766,1.582112,0.952915,1.124314
narrative_en,0.518270,0.962792,1.512703,0.877358,0.970806
prose_fr,0.872486,1.342679,1.616907,1.012625,1.085015


## 2. Stabilité du classement cross-taille

La question centrale : **les mêmes concepts sont-ils classés de la même façon d'une échelle à l'autre ?** Pour chaque métrique, on range les 5 concepts *dans* chaque échelle, puis on mesure la corrélation de rang de **Spearman** entre toutes les paires d'échelles. L'indice de stabilité est la moyenne de ces corrélations (1 = classement identique partout, 0 = aucun accord).

In [4]:
import itertools

def stability_index(metric, ascending):
    """Corrélation de Spearman moyenne des classements de concepts, sur paires d'echelles.
    Spearman = Pearson applique aux rangs (numpy, pas de scipy)."""
    ranks = {}
    for s in ORDER:
        sub = df[df["scale"] == s].groupby("concept")[metric].mean()
        ranks[s] = sub.rank(ascending=ascending)
    rhos = []
    for a, b in itertools.combinations(ORDER, 2):
        common = ranks[a].index.intersection(ranks[b].index)
        ra = ranks[a][common].to_numpy()
        rb = ranks[b][common].to_numpy()
        rhos.append(float(np.corrcoef(ra, rb)[0, 1]))
    return float(np.mean(rhos))

# sens : overlap10 plus haut = plus fidele (descending) ; rel_l2/kl plus bas = plus fidele (ascending)
stab = {
    "overlap10": stability_index("overlap10", ascending=False),
    "rel_l2":    stability_index("rel_l2",    ascending=True),
    "kl":        stability_index("kl",        ascending=True),
}
print("Indice de stabilite du classement de concepts (1 = identique dans toutes les echelles)")
for m, v in stab.items():
    print(f"  {m:10s} : {v:+.3f}")

Indice de stabilite du classement de concepts (1 = identique dans toutes les echelles)
  overlap10  : +0.730
  rel_l2     : +0.720
  kl         : +0.750


In [5]:
# Dans chaque echelle, l'ecart entre la couche profonde (0.50) et la superficielle (0.25)
delta = (df[df["frac_depth"] == 0.50].groupby("scale")[METRICS].mean()
         - df[df["frac_depth"] == 0.25].groupby("scale")[METRICS].mean())
delta = delta.reindex(ORDER)
print("Delta (profond 0.50  minus  superficiel 0.25) par echelle. "
      "Le SIGNE est-il stable cross-taille ?")
delta

Delta (profond 0.50  minus  superficiel 0.25) par echelle. Le SIGNE est-il stable cross-taille ?


,overlap10,rel_l2,kl
scale,,,
1.7B,0.015004,0.006742,-9.961708
2B,0.027682,-0.010502,-2.889779
8B,0.005486,0.015486,-4.075736
9B,0.033636,-0.013428,-5.091580
27B,0.013844,-0.306271,-1.531882


## 3. Lecture

Ce qui compte ici n'est pas la valeur absolue de la fidélité, mais **ce qui bouge et ce qui ne bouge pas** quand la taille change. Trois faits mesurés, repris en une vue d'ensemble :

In [6]:
# resume des 3 faits mesurés
print("=== 1) Tendance de la fidelite (moyenne par echelle) ===")
print(by_scale.round(4).to_string())
print("\n=== 2) Stabilite du classement de concepts (Spearman moyen) ===")
for m, v in stab.items():
    print(f"  {m:10s} : {v:+.3f}")
print("\n=== 3) Signe de l'ecart profondeur (profond - superficiel) par echelle ===")
print(np.sign(delta).to_string())

=== 1) Tendance de la fidelite (moyenne par echelle) ===
       overlap10  rel_l2       kl
scale                            
1.7B      0.0138  0.7678  17.4967
2B        0.0284  1.2303   9.6284
8B        0.0084  1.5905  18.9100
9B        0.0268  0.9646  11.9005
27B       0.0186  1.1144  11.1753

=== 2) Stabilite du classement de concepts (Spearman moyen) ===
  overlap10  : +0.730
  rel_l2     : +0.720
  kl         : +0.750

=== 3) Signe de l'ecart profondeur (profond - superficiel) par echelle ===
       overlap10  rel_l2   kl
scale                        
1.7B         1.0     1.0 -1.0
2B           1.0    -1.0 -1.0
8B           1.0     1.0 -1.0
9B           1.0    -1.0 -1.0
27B          1.0    -1.0 -1.0


## 3 (suite) — Lecture

Trois faits mesurés, sur les cinq échelles CPU :

1. **Le classement des concepts est substantiellement stable cross-taille.** L'indice de Spearman (classement des 5 concepts, moyen sur les paires d'échelles) tient à **0.72 – 0.75** pour les trois métriques — un accord fort, sans être parfait. Concrètement, `narrative_en` est le concept **le plus fidèle** (`rel_l2` le plus bas) **dans chacune des cinq échelles** ; `code_python` reste en queue de fidélité **aux cinq échelles** ; `prose_fr` y figure à 1.7B/2B/9B mais **s’en extraie à 8B (3ᵉ) et 27B (4ᵉ)**. Ce qui est lu de façon stable, c'est donc **quels** concepts, pas *à quel niveau*.

2. **La structure de profondeur est stable sur deux métriques sur trois.** Le signe de l'écart (couche 0.50 − couche 0.25) est **constant** pour `overlap10` (toujours + : la couche profonde capture plus de top-10) et pour `kl` (toujours − : la couche profonde est plus proche du modèle) — sur ces deux métriques et sur cet échantillon, la lecture J-Lens s'améliore en profondeur. Pour `rel_l2`, le signe **s'inverse** d'une échelle à l'autre : pas de direction stable.

3. **La magnitude de la fidélité n'est pas monotone en taille.** Ni `overlap10`, ni `rel_l2`, ni `kl` ne suit une tendance nette de 1.7B à 27B : elles oscillent. **C'est la réponse honnête au Concern 14B/27B du 16/11** : sur cet axe de taille, la fidélité ne « s'améliore pas » de façon prévisible avec la croissance du modèle — une divergence 14B/27B n'est donc **pas surprenante** au vu de ces cinq points.

**Limiter la portée.** Deux provenances de lens (1.7B/2B local-fit, 8B/9B/27B publiés) et `n_eval` = 4 : ce que ce carnet établit est une **observation sur ces cinq traces** : l'indice de stabilité repose sur le *classement* (peu sensible au bruit), le signe de profondeur sur une *différence de moyennes* (plus sensible) — **aucun test de sensibilité n'étant montré ici, ni l'un ni l'autre n'est encore une robustesse testée** ; les **niveaux** absolus de fidélité restent indicatifs. L'extension 14B/27B sur le même instrument (#27038) dira si l'accord 0.73 se maintient ou se dégrade franchement au-delà de 9B.

## Exercices

Trois prolongements de l'instrument. Chaque cellule code est un stub à compléter (pas d'erreur volontaire : le notebook s'exécute de bout en bout).

### Exercice 1 — Fidélité d'une échelle

Calculer la moyenne de `overlap10` pour une échelle donnée (argument `scale`).

In [7]:
def overlap_moyen(scale: str) -> float:
    """Moyenne de overlap10 sur tous les concepts et les 2 profondeurs d'une echelle.

    # TODO etudiant : filtrer df sur scale, puis moyenner la colonne 'overlap10'.
    """
    result = None  # TODO etudiant
    return result

### Exercice 2 — Classer les échelles pour un concept

Pour un concept donné, retourner les échelles triées de la **plus fidèle** à la **moins fidèle** selon `rel_l2` (le plus bas d'abord).

In [8]:
def echelles_par_fidelite(concept: str) -> list:
    """Echelles triees, du rel_l2 le plus bas (fidele) au plus haut, pour un concept.

    # TODO etudiant : filtrer df sur concept, moyenner rel_l2 par scale, trier.
    """
    result = None  # TODO etudiant
    return result

### Exercice 3 — Stabilité d'une métrique

Réutiliser la logique de `stability_index` pour retourner l'indice de stabilité d'une métrique donnée, avec le sens de tri approprié.

In [9]:
def indice_stabilite(metric: str) -> float:
    """Indice de stabilite (Spearman moyen) pour une metrique, sens adapte.

    # Indice : overlap10 est trié descending, rel_l2/kl sont triés ascending.
    # TODO etudiant : determiner le sens puis appeler stability_index.
    """
    result = None  # TODO etudiant
    return result

## Conclusion

L'instrument ICT-49 transforme la question « la lecture J-Lens est-elle stable quand le modèle grossit ? » en **mesures reproductibles** sur les cinq échelles CPU. La suite — 14B/27B sur le même instrument et l'analyse de la **dynamique** des catastrophes sous perturbation — est portée par l'extension GPU (#27038) et l'EPIC #8236.

## À retenir

- **La question se mesure.** « La lecture J-Lens est-elle stable quand le modèle grossit ? » devient **des mesures reproductibles** sur les cinq échelles CPU déjà commitées : la fidélité de lecture et la structure de profondeur, suivies d'une échelle en taille.
- **Le classement tient, la magnitude non.** Quand la taille monte, le **classement des concepts** lu par J-Lens est stable (Spearman ~0,72–0,75) — mais la **magnitude** de la fidélité n'est **pas monotone** en taille. Stabilité de l'ordre ≠ croissance de la précision.
- **Le sens de l'indice dépend de la métrique.** `overlap10` est triée **décroissante**, `rel_l2` et `kl` **croissantes** : un indice de stabilité a un *sens* propre à chaque métrique, et le mélanger invertit la lecture.


## Vérifiez votre compréhension

1. `overlap10` est triée **décroissante** mais `rel_l2`/`kl` **croissantes** : que doit faire `indice_stabilite` pour que l'indice de Spearman moyen ait un sens sur les deux familles de métriques ?
2. Que distingue ce carnet entre « le **classement** des concepts est stable » et « la **magnitude** de la fidélité croît » ? Pourquoi l'une des deux est-elle le résultat porteur ?
3. Sur quelles **cinq échelles** CPU l'instrument est-il calibré, et quelle extension en porte la suite en taille ?


## Pour aller plus loin

- **La suite en taille** — 14B/27B sur le **même instrument** — est portée par l'extension GPU ([#27038](https://github.com/jsboige/CoursIA/issues/27038)).
- **La dynamique des catastrophes** sous perturbation, au-delà de la fidélité statique, est portée par l'**EPIC #8236**.
- **Les carnets frères** : le tête-à-tête SAE↔J-Lens (ICT-24) et la géométrie factored (ICT-36) — ce carnet leur ajoute l'axe de la **taille du modèle**.
